# Lab 04: Tool Use & Function Calling with NVIDIA NIMs

**NCP-AAI Domain:** Agent Development (5-8%)

## What You'll Build
A production-ready tool-use agent with:
- OpenAI-compatible function calling via NVIDIA NIMs
- Multiple tools: database query, calculator, weather API
- Parallel tool calls (multiple tools in one turn)
- Error handling: malformed args, tool failures, retries
- Structured output with response_format

## Learning Objectives
- Define tools using the OpenAI function schema format
- Handle the tool message role (not user/assistant)
- Manage parallel tool calls
- Implement retry logic for malformed tool arguments
- Use temperature=0 for deterministic tool selection

In [ ]:
!pip install -q openai langchain-nvidia-ai-endpoints tenacity

In [ ]:
import os
os.environ['NVIDIA_API_KEY'] = 'nvapi-xxx'  # Replace with your key

## Step 1: Define Tools Using OpenAI Schema

In [ ]:
# The OpenAI function schema format — used by NVIDIA NIMs
# This is what the exam tests: name, description, parameters structure
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_gpu_specs",
            "description": "Get technical specifications for an NVIDIA GPU model including VRAM, compute capability, and supported quantization formats.",
            "parameters": {
                "type": "object",
                "properties": {
                    "gpu_model": {
                        "type": "string",
                        "description": "The GPU model name, e.g. 'H100', 'A100', 'RTX 4090'"
                    },
                    "include_pricing": {
                        "type": "boolean",
                        "description": "Whether to include cloud pricing estimates",
                        "default": False
                    }
                },
                "required": ["gpu_model"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_tokens",
            "description": "Calculate the number of tokens in a text string for a given tokenizer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The text to tokenize and count"
                    },
                    "model": {
                        "type": "string",
                        "description": "Model name to use its tokenizer",
                        "enum": ["llama-3", "mistral", "gpt-4"]
                    }
                },
                "required": ["text", "model"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_inference_cost",
            "description": "Estimate inference cost in USD for a given number of tokens and model.",
            "parameters": {
                "type": "object",
                "properties": {
                    "input_tokens": {"type": "integer", "description": "Number of input tokens"},
                    "output_tokens": {"type": "integer", "description": "Number of output tokens"},
                    "model": {"type": "string", "description": "Model name for pricing lookup"}
                },
                "required": ["input_tokens", "output_tokens", "model"]
            }
        }
    }
]

print(f'Defined {len(tools)} tools')
for t in tools:
    print(f'  - {t["function"]["name"]}: {t["function"]["description"][:60]}...')

## Step 2: Implement Tool Functions

In [ ]:
import json

# Mock tool implementations (production would call real APIs/DBs)
GPU_DB = {
    'H100': {
        'vram': '80GB HBM3',
        'compute_capability': 9.0,
        'quantization': ['FP8', 'INT8 SmoothQuant', 'INT4 AWQ'],
        'in_flight_batching': True,
        'cloud_price_hr': '$3.50 (A100-equivalent)'
    },
    'A100': {
        'vram': '80GB HBM2e',
        'compute_capability': 8.0,
        'quantization': ['INT8 SmoothQuant', 'INT4 AWQ'],
        'in_flight_batching': True,
        'cloud_price_hr': '$2.21'
    },
    'RTX 4090': {
        'vram': '24GB GDDR6X',
        'compute_capability': 8.9,
        'quantization': ['FP8', 'INT8 SmoothQuant', 'INT4 AWQ'],
        'in_flight_batching': False,
        'cloud_price_hr': 'Consumer GPU - varies'
    }
}

PRICING = {
    'meta/llama-3.1-70b-instruct': {'input': 0.00035, 'output': 0.00040},  # per 1K tokens
    'meta/llama-3.1-8b-instruct': {'input': 0.00005, 'output': 0.00008},
    'nvidia/llama-3.1-nemotron-70b-instruct': {'input': 0.00040, 'output': 0.00045},
}

def get_gpu_specs(gpu_model: str, include_pricing: bool = False) -> dict:
    gpu = GPU_DB.get(gpu_model.upper(), GPU_DB.get(gpu_model))
    if not gpu:
        return {'error': f'GPU {gpu_model} not found. Available: {list(GPU_DB.keys())}'}
    result = dict(gpu)
    if not include_pricing:
        result.pop('cloud_price_hr', None)
    return result

def calculate_tokens(text: str, model: str) -> dict:
    # Approximate: ~4 chars per token for English text
    token_count = len(text) // 4
    return {'token_count': token_count, 'model': model, 'text_length': len(text)}

def estimate_inference_cost(input_tokens: int, output_tokens: int, model: str) -> dict:
    pricing = PRICING.get(model)
    if not pricing:
        return {'error': f'Pricing not found for {model}', 'available_models': list(PRICING.keys())}
    cost = (input_tokens / 1000 * pricing['input']) + (output_tokens / 1000 * pricing['output'])
    return {
        'total_cost_usd': round(cost, 6),
        'input_cost': round(input_tokens / 1000 * pricing['input'], 6),
        'output_cost': round(output_tokens / 1000 * pricing['output'], 6)
    }

TOOL_FUNCTIONS = {
    'get_gpu_specs': get_gpu_specs,
    'calculate_tokens': calculate_tokens,
    'estimate_inference_cost': estimate_inference_cost,
}

print('Tool functions ready')
# Quick test
print(get_gpu_specs('H100', include_pricing=True))

## Step 3: Tool-Calling Agent Loop

In [ ]:
from openai import OpenAI
import json

# NVIDIA NIMs via OpenAI-compatible client
client = OpenAI(
    base_url='https://integrate.api.nvidia.com/v1',
    api_key=os.environ['NVIDIA_API_KEY']
)

def run_tool_agent(user_query: str, max_iterations: int = 5) -> str:
    """Complete tool-calling agent loop with error handling."""
    messages = [
        {"role": "system", "content": "You are an NVIDIA infrastructure assistant. Use tools to answer questions accurately. Be concise."},
        {"role": "user", "content": user_query}
    ]

    for iteration in range(max_iterations):
        print(f'  [Iteration {iteration+1}]')

        # Call LLM with tools
        response = client.chat.completions.create(
            model='meta/llama-3.1-70b-instruct',
            messages=messages,
            tools=tools,
            tool_choice='auto',   # LLM decides when to use tools
            temperature=0.0,      # Deterministic tool selection
        )

        msg = response.choices[0].message
        messages.append(msg)  # Add assistant message to history

        # Check if LLM made tool calls
        if not msg.tool_calls:
            # No tool calls = LLM has a final answer
            print(f'  Final answer generated')
            return msg.content

        # Execute each tool call
        # IMPORTANT: tool result goes in 'tool' role with tool_call_id
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            print(f'  Calling tool: {func_name}')

            try:
                # Parse tool arguments
                args = json.loads(tool_call.function.arguments)

                # Execute tool
                if func_name in TOOL_FUNCTIONS:
                    result = TOOL_FUNCTIONS[func_name](**args)
                else:
                    result = {'error': f'Tool {func_name} not found'}

            except json.JSONDecodeError as e:
                result = {'error': f'Invalid tool arguments JSON: {e}'}
            except TypeError as e:
                result = {'error': f'Wrong argument types: {e}'}
            except Exception as e:
                result = {'error': f'Tool execution failed: {e}'}

            # Add tool result to messages (MUST use role='tool')
            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,  # Links result to specific call
                'name': func_name,
                'content': json.dumps(result)
            })

    return 'Max iterations reached without final answer'

# Test
result = run_tool_agent('What VRAM does the H100 have and which quantization formats does it support?')
print(f'\nFinal Answer: {result}')

## Step 4: Parallel Tool Calls

In [ ]:
# LLMs can request multiple tool calls in a SINGLE response
# This happens when multiple independent pieces of information are needed

result = run_tool_agent(
    'Compare H100 and A100 GPUs for quantization support, and estimate the cost of \
     running 1000 input tokens + 500 output tokens on llama-3.1-70b-instruct.'
)
print(f'\nFinal Answer: {result}')

## Step 5: Structured Output with response_format

In [ ]:
# Structured output: forces valid JSON matching a schema
# Critical for reliable downstream parsing in agent pipelines

response = client.chat.completions.create(
    model='meta/llama-3.1-70b-instruct',
    messages=[
        {"role": "system", "content": "Return GPU analysis as JSON."},
        {"role": "user", "content": "Analyze H100 GPU for production LLM serving. Return JSON with: gpu_name, vram, recommended_quantization, max_batch_size_estimate, notes"}
    ],
    response_format={"type": "json_object"},  # Guarantees valid JSON output
    temperature=0.0,
)

result_json = json.loads(response.choices[0].message.content)
print('Structured output (guaranteed valid JSON):')
print(json.dumps(result_json, indent=2))

## Exam Recap

**OpenAI tool schema structure:**
```json
{"type": "function", "function": {"name": "...", "description": "...", "parameters": {"type": "object", "properties": {...}, "required": [...]}}}
```

**Tool result message format:**
```json
{"role": "tool", "tool_call_id": "call_abc123", "name": "tool_name", "content": "result string"}
```

| Setting | Value | Why |
|---------|-------|-----|
| temperature | 0.0 | Deterministic tool selection |
| response_format | json_object | Guaranteed valid JSON |
| tool_choice | auto | LLM decides when to use tools |

## Challenge Exercises
1. Add a 4th tool: `search_documentation(query)` — make the agent use it for unknown questions
2. Simulate a tool failure (return HTTP 500 error) — add retry logic with tenacity
3. Track total token usage across all iterations; alert if cost exceeds $0.01
4. Force parallel tool calls: ask a question that clearly needs 3 tools simultaneously